In [ ]:
import sys
import os
package_path = os.path.abspath("..")  
sys.path.insert(0, package_path)
#The path will be managed by conda or whatever on release, but this is fine for now.

import scMPRAforge as scm

#load the autoreload extension
%load_ext autoreload
#reload code on every execution
#(this may break objects)
#you can remove this & do dev in a notebook, then paste into the module when you are done.
%autoreload 2

scm.helloworld()

hello world!


In [1]:
import pandas as pd
import numpy as np

In [2]:
dummy=pd.DataFrame({"bc":["AATC","AATC","ATGC","ATGC","AATG","TTGC"],"other_info":["foo","bar","baz","biff","bom","bim"]})
dummy

,bc,other_info
0,AATC,foo
1,AATC,bar
2,ATGC,baz
3,ATGC,biff
4,AATG,bom
5,TTGC,bim


row 4 is a seq error of row 0/1, row 5 is a seq error of row 2/3. 

In [3]:
scm.flatten_barcode_errors(dummy,barcode_column="bc")

,bc,other_info
0,AATC,foo
1,AATC,bar
2,ATGC,baz
3,ATGC,biff
4,AATC,bom
5,ATGC,bim


perfect


In [2]:
barcode_len=16
number_of_cells=10000

nts=[0,1,2,3]#A T G C

rng = np.random.default_rng(seed=42)

#make a big block of ground-truth barcodes
bcs=rng.choice(nts,barcode_len*number_of_cells)
bcs.shape=(number_of_cells,barcode_len)

#make sure there are no duplicate barcodes (low chance)
assert not np.any(np.unique(bcs,axis=1,return_counts=True)[1] >1)

#multiple instances of each barcode (multiple reads)
umis_per_cell=100
reads=np.repeat(bcs,umis_per_cell,axis=0)

#for each nucleotide: does it have an error?
chance_of_error=1/200#per_nt
error_bool=rng.binomial(1,chance_of_error,size=barcode_len*number_of_cells*umis_per_cell)
error_bool.shape=(number_of_cells*umis_per_cell,barcode_len)

#if there is an error, how many nt to advance by?
#from 1 up to one less than all the way around
to_advance=rng.choice(range(1,len(nts)),barcode_len*number_of_cells*umis_per_cell)
to_advance.shape=(number_of_cells*umis_per_cell,barcode_len)

print("mutating")
#multiply to get the actual amount to mutate by
reads=((to_advance*error_bool)+reads) % 4

print("mapping")
# Convert each row into a string of corresponding characters
mapping = np.array(['A', 'T', 'G', 'C'])
bc_series = mapping[reads]
print("joining")
bc_series=row_strings = np.apply_along_axis(lambda row: ''.join(map(str, row)), axis=1, arr=bc_series)
bc_series=pd.Series(bc_series)


mutating
mapping
joining


In [3]:
bc_series

0         ACGTTCAGAAGCGCGC
1         ACGTTCAGAAGCGGGC
2         ACGTTCAGAAGCGCGC
3         ACGTTCAGAGGCGCGC
4         ACGTTCAGAAGCGCGC
                ...       
999995    TACTGTACTAGACCGG
999996    TACTGTACTAGACCGG
999997    TACTGTACTAGACCGG
999998    TACTGTACTAGACCGG
999999    TACTGTACTAGACCGG
Length: 1000000, dtype: object

In [32]:
len(bc_series.unique())

8157

In [5]:
df=pd.DataFrame({'bc':bc_series})

In [44]:
import cProfile
import pstats


In [45]:
profiler = cProfile.Profile()
profiler.enable()
scm.flatten_barcode_errors(df,"bc")
profiler.disable()

stats = pstats.Stats(profiler)


#cProfile.run('scm.flatten_barcode_errors(df,"bc")')


In [ ]:
stats.sort_stats("cumulative").print_stats(10)  # Top 10 slowest functions

In [ ]:
stats.print_callers("get")

In [ ]:
import trace
tracer = trace.Trace(trace=True, count=False)
tracer.run("scm.flatten_barcode_errors(df,'bc')")  # Replace with your function

In [ ]:
from line_profiler import LineProfiler

def wrapper():
    scm.flatten_barcode_errors(df,"bc")

lp = LineProfiler()
lp.add_function(wrapper)
lp.enable()
wrapper()
lp.disable()
lp.print_stats()

In [6]:
scm.flatten_barcode_errors(df,"bc")

Clustering execution time: 65.371517 seconds


,bc
0,ACGTTCAGAAGCGCGC
1,ACGTTCAGAAGCGCGC
2,ACGTTCAGAAGCGCGC
3,ACGTTCAGAAGCGCGC
4,ACGTTCAGAAGCGCGC
...,...
999995,TACTGTACTAGACCGG
999996,TACTGTACTAGACCGG
999997,TACTGTACTAGACCGG
999998,TACTGTACTAGACCGG
